In [0]:
%sql
use catalog delta_lake_study_demo;


In [0]:
%sql
use schema delta_lake_demo;

In [0]:
%sql
drop table orders_timetravel

In [0]:
%sql
create or replace table orders_timetravel (
    order_id bigint,
    sku string,
    product_name string,
    product_category string,
    qty int,
    unit_price decimal(10,2)
)using DELTA
LOCATION 'abfss://externaldata@deltalakelearnstg.dfs.core.windows.net/orders_timetravel'

In [0]:
%sql
insert into delta_lake_study_demo.delta_lake_demo.orders_timetravel values (1,'A101','iPhone Charger','Electronics',2,19.99),(2,'A102','Bluetooth Headphones','Electronics',1,49.99),(3,'A103','Apple Watch','Electronics',1,399.99),(4,'A104','AirPods','Electronics',1,149.99),(5,'A105','MacBook Pro','Electronics',1,1999.99);
    

In [0]:
%sql
create or replace temp view incoming_source as
select * from values (1,'A111','iPhone','Electronics',2,19.99),(2,'A102','Bluetooth Earpods','Electronics',1,49.99),(3,'A103','Apple Watch','Electronics',1,399.99),(4,'A114','Mouse','Electronics',1,149.99),(6,'A106','MacBook','Electronics',1,1999.99) as t(order_id,sku,product_name, product_category,qty,unit_price)

In [0]:
%sql
describe history orders_timetravel

In [0]:
%sql
select * from delta.`abfss://externaldata@deltalakelearnstg.dfs.core.windows.net/orders_timetravel`

In [0]:
%sql
merge into orders_timetravel tgt
using incoming_source src
on tgt.order_id = src.order_id
when matched then 
update set tgt.order_id = src.order_id, tgt.product_name = src.product_name, tgt.sku = src.sku, tgt.product_category = src.product_category,
tgt.qty = src.qty, tgt.unit_price = src.unit_price
when not matched then insert *

In [0]:
%sql
select * from delta.`abfss://externaldata@deltalakelearnstg.dfs.core.windows.net/orders_timetravel`

In [0]:
%sql
merge into orders_timetravel tgt
using incoming_source src
on tgt.order_id = src.order_id
when matched then 
update set *
when not matched then insert *

In [0]:
%sql
select * from delta.`abfss://externaldata@deltalakelearnstg.dfs.core.windows.net/orders_timetravel`

In [0]:
%sql
describe history orders_timetravel

In [0]:
%sql
select * from json.`abfss://externaldata@deltalakelearnstg.dfs.core.windows.net/orders_timetravel/_delta_log/00000000000000000004.json`

In [0]:
%sql
create or replace temp view incoming_source as
select * from values (1,'A111','iPhone','Electronics',2,19.99,FALSE),(2,'A102','Bluetooth Earpods','Electronics',1,49.99,TRUE),(3,'A103','Apple Watch','Electronics',1,399.99,FALSE),(4,'A114','Mouse','Electronics',1,149.99,FALSE),(6,'A106','MacBook','Electronics',1,1999.99,TRUE) as t(order_id,sku,product_name, product_category,qty,unit_price,is_deleted)

In [0]:
%sql
select * from delta.`abfss://externaldata@deltalakelearnstg.dfs.core.windows.net/orders_timetravel`

In [0]:
%sql
merge into orders_timetravel as tgt
using incoming_source as src
on tgt.order_id = src.order_id
when matched and is_deleted = TRUE then delete 
when matched then update set *
when not matched then insert * 

In [0]:
%sql
select * from delta.`abfss://externaldata@deltalakelearnstg.dfs.core.windows.net/orders_timetravel`

In [0]:
%sql
create or replace temp view incoming_source as
select * from values (1,'A111','iPhone','Electronics',2,19.99),(2,'A102','Bluetooth Earpods','Electronics',1,49.99),(3,'A103','Apple Watch','Electronics',1,399.99) as t(order_id,sku,product_name, product_category,qty,unit_price)

In [0]:
%sql
merge into orders_timetravel as tgt
using incoming_source as src
on tgt.order_id = src.order_id
when matched then update set *
when not matched then insert *
when not matched by source then delete 

In [0]:
%sql
select * from delta.`abfss://externaldata@deltalakelearnstg.dfs.core.windows.net/orders_timetravel`